# Phase 1: Automotive Document Data Ingestion and Processing

Welcome! This notebook demonstrates the foundation of our Question Answering system.
In Phase 1, our goal is to convert messy raw documents (PDFs, Images, Excel sheets) into clean, bite-sized facts that the AI can rapidly search through.

### What to expect:
1. **Ingestion**: Loading files.
2. **Cleaning & Metadata**: Stripping garbage text and adding structured labels.
3. **Chunking**: Cutting documents into overlapping 500-character blocks.
4. **Embedding**: Using a SentenceTransformer model to convert text into math vectors.
5. **Indexing**: Building our offline, CPU-friendly FAISS database.

In [ ]:
!pip install -r ../requirements.txt
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
from src.processing import process_document
from src.chunking import recursive_chunking
from src.embeddings import generate_embeddings
from src.vector_store import VectorStore

print("Libraries successfully loaded. Ready to begin Phase 1 pipeline!")

## 1. Load and Process Documents
Watch as the system automatically extracts raw text and adds metadata based on our folder structure.

In [ ]:
file_path = '../test_assets/sample.pdf'
text, metadata = process_document(file_path)

print("--- Extraction Complete ---")
print(f"Document Metadata: {metadata}")

## 2. Text Chunking
AI cannot read thousands of pages at once. We break the text into 500 character chunks.

In [ ]:
chunks = recursive_chunking(text)
metadatas = [metadata.copy() for _ in chunks]

print(f"Document chopped into {len(chunks)} overlapping chunks.")

## 3. Embedding and Indexing
Finally, we run these chunks through our lightweight offline model (`all-MiniLM-L6-v2`) and save them into the FAISS vector database. Look for the `tqdm` progress bars!

In [ ]:
embeddings = generate_embeddings(chunks)

vs = VectorStore()
vs.build_index(embeddings, chunks, metadatas)
vs.save_index('../faiss_index')

print("✅ Phase 1 Successful! Database built and saved to disk.")